In [ ]:
"""
Fuzzy C-Means Clustering
Analisis Distribusi Kasus Penyakit Campak di Sumatera Barat
Kelompok 3 - Perancangan dan Analisis Algoritma
"""

import numpy as np

# ============================================================
# DATA
# ============================================================
data_kasus = {
    "Kabupaten Agam": 6,
    "Kabupaten Kepulauan Mentawai": 1,
    "Kota Padang": 5,
    "Kabupaten Pasaman Barat": 4,
    "Kabupaten Solok Selatan": 3,
    "Kota Pariaman": 5,
    "Kabupaten Sijunjung": 1,
    "Kabupaten Tanah Datar": 2,
    "Kota Sawahlunto": 4,
    "Kabupaten Pesisir Selatan": 4,
    "Kota Bukittinggi": 3,
    "Kabupaten Dharmasraya": 1,
    "Kota Padang Panjang": 9,
    "Kabupaten Lima Puluh Kota": 2,
    "Kabupaten Pasaman": 5,
    "Kota Payakumbuh": 4,
    "Kabupaten Padang Pariaman": 3,
    "Kabupaten Solok": 2,
}

wilayah      = list(data_kasus.keys())
kasus        = np.array(list(data_kasus.values()), dtype=float)
label_cluster = ["Rendah", "Sedang", "Tinggi"]

# ============================================================
# PARAMETER FCM (sesuai laporan)
# ============================================================
C         = 3
m         = 2
MAX_ITER  = 100
EPSILON   = 1e-6
centroid  = np.array([2.0, 5.0, 8.0])   # C1=rendah, C2=sedang, C3=tinggi

# ============================================================
# FUNGSI FCM
# ============================================================
def hitung_keanggotaan(kasus_arr, centroid):
    n  = len(kasus_arr)
    C  = len(centroid)
    U  = np.zeros((n, C))
    ex = 2.0 / (m - 1)
    for i in range(n):
        d = np.abs(kasus_arr[i] - centroid)
        zi = np.where(d == 0)[0]
        if len(zi):
            U[i, zi[0]] = 1.0
        else:
            for j in range(C):
                U[i, j] = 1.0 / np.sum((d[j] / d) ** ex)
    return U

def update_centroid(kasus_arr, U):
    C  = U.shape[1]
    c  = np.zeros(C)
    for j in range(C):
        um = U[:, j] ** m
        c[j] = np.sum(um * kasus_arr) / np.sum(um)
    return c

# ============================================================
# JALANKAN ITERASI FCM (tersembunyi, hanya untuk dapat centroid final)
# ============================================================
for _ in range(MAX_ITER):
    c_lama   = centroid.copy()
    U        = hitung_keanggotaan(kasus, centroid)
    centroid = update_centroid(kasus, U)
    if np.max(np.abs(centroid - c_lama)) < EPSILON:
        break

U_final = hitung_keanggotaan(kasus, centroid)

# ============================================================
# FUNGSI TAMPILKAN PERHITUNGAN LENGKAP
# ============================================================
WARNA = {"Rendah": "\033[92m", "Sedang": "\033[93m", "Tinggi": "\033[91m"}
RESET = "\033[0m"
BOLD  = "\033[1m"

def tampilkan_perhitungan(nama):
    idx = wilayah.index(nama)
    x   = kasus[idx]
    u   = U_final[idx]
    cl  = label_cluster[np.argmax(u)]
    w   = WARNA[cl]

    print("\n" + "=" * 56)
    print(f"  HASIL FUZZY C-MEANS")
    print(f"  Wilayah : {BOLD}{nama}{RESET}")
    print(f"  Jumlah kasus positif & terkonfirmasi : {int(x)}")
    print("=" * 56)

    c_awal = np.array([2.0, 5.0, 8.0])

    print("\n  [1] PARAMETER")
    print(f"      Jumlah cluster (C) = {C}")
    print(f"      Fuzziness (m)      = {m}")
    print(f"      Centroid           : C1=2 (Rendah), C2=5 (Sedang), C3=8 (Tinggi)")

    print("\n  [2] HITUNG JARAK  d = |x - c|")
    d = np.abs(x - c_awal)
    d_tampil = np.where(d == 0, 1.0, d)
    for j, (lb, cv) in enumerate(zip(label_cluster, c_awal)):
        hasil = int(d_tampil[j]) if d_tampil[j] == int(d_tampil[j]) else d_tampil[j]
        print(f"      ke C{j+1} ({lb:6s}) : |{int(x)} - {int(cv)}| = {hasil}")

    print("\n  [3] HITUNG DERAJAT KEANGGOTAAN")
    print(f"      Rumus : u_ij = 1 / Σ (d_ij / d_ik)^(2/(m-1))")
    u_show = hitung_keanggotaan(np.array([x]), c_awal)[0]
    for j, lb in enumerate(label_cluster):
        pembagi = sum((d_tampil[j]/d_tampil[k])**2 for k in range(C))
        print(f"      u_{lb:6s} = 1 / {pembagi:.4f} = {u_show[j]:.2f}  ({u_show[j]*100:.1f}%)")

    print("\n  [4] VISUALISASI KEANGGOTAAN")
    BAR = 30
    for j, lb in enumerate(label_cluster):
        filled = int(round(u_show[j] * BAR))
        bar    = "█" * filled + "░" * (BAR - filled)
        wc     = WARNA[lb]
        print(f"      {lb:6s}  {wc}{bar}{RESET}  {u_show[j]*100:5.1f}%")

    cl_show = label_cluster[np.argmax(u_show)]
    w_show  = WARNA[cl_show]
    print(f"\n  [5] KESIMPULAN")
    print(f"      {BOLD}{w_show}➜ {nama} masuk Cluster {cl_show.upper()}{RESET}")
    print(f"         Keanggotaan tertinggi pada cluster {cl_show} = {max(u_show)*100:.1f}%")
    print("=" * 56)

# ============================================================
# INPUT DAERAH DI AWAL — baru keluar perhitungan
# ============================================================
print("=" * 70)
print("  FUZZY C-MEANS — KASUS CAMPAK SUMATERA BARAT")
print("=" * 70)
print()
print(f"  {'No':<4} {'Kabupaten/Kota':<35} {'Jumlah Kasus Yang Positif':>25}")
print(f"  {'':4} {'':35} {'dan Terkonfirmasi Positif Campak':>25}")
print("  " + "-" * 66)
for i, (w, k) in enumerate(zip(wilayah, kasus), 1):
    print(f"  {i:<4} {w:<35} {int(k):>25}")
print("  " + "-" * 66)
print()
print("=" * 70)
print("\n  Daftar wilayah yang tersedia:")
for i, w in enumerate(wilayah, 1):
    print(f"  {i:>2}. {w}")

print("\n  Ketik nama daerah (atau sebagian nama) untuk")
print("  melihat perhitungan cluster FCM-nya.")
print("  Ketik 'keluar' untuk berhenti.\n")

while True:
    try:
        query = input("  Masukkan nama daerah: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n  Program dihentikan.")
        break

    if query.lower() in ("keluar", "exit", "quit", "q"):
        print("  Sampai jumpa!")
        break

    if not query:
        continue

    cocok = [w for w in wilayah if query.lower() in w.lower()]

    if not cocok:
        print(f"\n  ✗ '{query}' tidak ditemukan. Coba kata kunci lain.\n")

    elif len(cocok) == 1:
        tampilkan_perhitungan(cocok[0])
        print()

    else:
        print(f"\n  Ditemukan {len(cocok)} wilayah:")
        for i, h in enumerate(cocok, 1):
            print(f"  {i}. {h}")
        try:
            pilih = input("  Pilih nomor: ").strip()
            if pilih.isdigit() and 1 <= int(pilih) <= len(cocok):
                tampilkan_perhitungan(cocok[int(pilih) - 1])
            else:
                print("  Nomor tidak valid.")
        except (EOFError, KeyboardInterrupt):
            break
        print()

# ============================================================
# VISUALISASI SEMUA DAERAH
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

warna_map = {"Rendah": "#639922", "Sedang": "#EF9F27", "Tinggi": "#E24B4A"}
bg_map    = {"Rendah": "#EAF3DE", "Sedang": "#FAEEDA", "Tinggi": "#FCEBEB"}

cluster_index = np.argmax(U_final, axis=1)
cluster_label = [label_cluster[i] for i in cluster_index]
warna_titik   = [warna_map[c] for c in cluster_label]

sorted_idx    = np.argsort(kasus)
s_wilayah     = [wilayah[i] for i in sorted_idx]
s_kasus       = kasus[sorted_idx]
s_cluster     = [cluster_label[i] for i in sorted_idx]
s_warna       = [warna_map[c] for c in s_cluster]
s_U           = U_final[sorted_idx]

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor("#FAFAFA")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Bar chart jumlah kasus ──────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor("#F8F8F8")
bars = ax1.barh(range(len(s_wilayah)), s_kasus, color=s_warna,
                edgecolor="white", height=0.65)
ax1.set_yticks(range(len(s_wilayah)))
ax1.set_yticklabels(
    [w.replace("Kabupaten ", "Kab. ") for w in s_wilayah], fontsize=9)
ax1.set_xlabel("Jumlah Kasus Positif & Terkonfirmasi", fontsize=10)
ax1.set_title("Jumlah Kasus Campak per Wilayah — Sumatera Barat",
              fontsize=12, fontweight="bold", pad=10)
ax1.set_xlim(0, 11)
ax1.grid(axis="x", linestyle="--", alpha=0.4)
ax1.spines[["top","right"]].set_visible(False)

for j, (c, lbl) in enumerate(zip(centroid, label_cluster)):
    ax1.axvline(x=c, color=warna_map[lbl], linestyle="--",
                linewidth=1.2, alpha=0.7,
                label=f"Centroid {lbl} ({c:.2f})")

for bar, val, cl in zip(bars, s_kasus, s_cluster):
    ax1.text(val + 0.08, bar.get_y() + bar.get_height()/2,
             f"{int(val)}  [{cl}]",
             va="center", fontsize=8.5,
             color=warna_map[cl], fontweight="bold")

patches = [mpatches.Patch(color=warna_map[lb], label=f"Cluster {lb}")
           for lb in label_cluster]
ax1.legend(handles=patches + [
    plt.Line2D([0],[0], color=warna_map[lb], linestyle="--", linewidth=1.2,
               label=f"Centroid {lb} ({c:.2f})")
    for lb, c in zip(label_cluster, centroid)
], fontsize=8, loc="lower right")

# ── Plot 2: Stacked bar derajat keanggotaan ──────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor("#F8F8F8")
y    = np.arange(len(s_wilayah))
left = np.zeros(len(s_wilayah))
for j, lb in enumerate(label_cluster):
    vals = s_U[:, j] * 100
    ax2.barh(y, vals, left=left, color=warna_map[lb],
             edgecolor="white", height=0.65, label=lb)
    for k, (v, l) in enumerate(zip(vals, left)):
        if v > 8:
            ax2.text(l + v/2, k, f"{v:.0f}%",
                     ha="center", va="center", fontsize=7,
                     color="white", fontweight="bold")
    left += vals

ax2.set_yticks(y)
ax2.set_yticklabels(
    [w.replace("Kabupaten ", "Kab. ") for w in s_wilayah], fontsize=8.5)
ax2.set_xlabel("Derajat Keanggotaan (%)", fontsize=10)
ax2.set_xlim(0, 100)
ax2.set_title("Derajat Keanggotaan Fuzzy per Wilayah",
              fontsize=11, fontweight="bold", pad=8)
ax2.legend(fontsize=8, loc="lower right")
ax2.spines[["top","right"]].set_visible(False)
ax2.grid(axis="x", linestyle="--", alpha=0.3)

# ── Plot 3: Scatter keanggotaan tertinggi vs kasus ───────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor("#F8F8F8")
for i in range(len(wilayah)):
    u_max = np.max(U_final[i])
    cl    = cluster_label[i]
    ax3.scatter(kasus[i], u_max * 100,
                color=warna_map[cl], s=120, zorder=5,
                edgecolors="white", linewidth=1.2)
    short = (wilayah[i]
             .replace("Kabupaten ", "Kab. ")
             .replace("Kota ", ""))
    ax3.annotate(short, (kasus[i], u_max * 100),
                 textcoords="offset points", xytext=(6, 3),
                 fontsize=7.5, color="#444")

for j, (c, lb) in enumerate(zip(centroid, label_cluster)):
    ax3.axvline(x=c, color=warna_map[lb], linestyle=":",
                linewidth=1.2, alpha=0.6)

ax3.set_xlabel("Jumlah Kasus", fontsize=10)
ax3.set_ylabel("Keanggotaan Tertinggi (%)", fontsize=10)
ax3.set_title("Scatter: Keanggotaan Tertinggi vs Jumlah Kasus",
              fontsize=11, fontweight="bold", pad=8)
ax3.set_xlim(0, 11)
ax3.set_ylim(40, 105)
ax3.grid(True, linestyle="--", alpha=0.3)
ax3.spines[["top","right"]].set_visible(False)

patches2 = [mpatches.Patch(color=warna_map[lb], label=f"Cluster {lb}")
            for lb in label_cluster]
ax3.legend(handles=patches2, fontsize=8)

fig.suptitle(
    "Fuzzy C-Means Clustering — Distribusi Kasus Campak Sumatera Barat\n"
    f"C={C}  |  m={m}  |  Centroid final: "
    f"Rendah={centroid[0]:.2f}, Sedang={centroid[1]:.2f}, Tinggi={centroid[2]:.2f}",
    fontsize=13, fontweight="bold", y=0.98)

plt.savefig("/mnt/user-data/outputs/fuzzy_c_means_visualisasi.png",
            dpi=150, bbox_inches="tight", facecolor="#FAFAFA")
plt.show()
print("\n✓ Visualisasi disimpan: fuzzy_c_means_visualisasi.png")

  FUZZY C-MEANS — KASUS CAMPAK SUMATERA BARAT

  No   Kabupaten/Kota                      Jumlah Kasus Yang Positif
                                           dan Terkonfirmasi Positif Campak
  ------------------------------------------------------------------
  1    Kabupaten Agam                                              6
  2    Kabupaten Kepulauan Mentawai                                1
  3    Kota Padang                                                 5
  4    Kabupaten Pasaman Barat                                     4
  5    Kabupaten Solok Selatan                                     3
  6    Kota Pariaman                                               5
  7    Kabupaten Sijunjung                                         1
  8    Kabupaten Tanah Datar                                       2
  9    Kota Sawahlunto                                             4
  10   Kabupaten Pesisir Selatan                                   4
  11   Kota Bukittinggi                          